# 💰 Income Evaluation Classification — Random Forest (Ensemble)

**🇹🇷 Türkçe:**
Bu notebook, UCI Adult / Census Income veri setini kullanarak bir kişinin yıllık gelirinin 50.000 doların üzerinde mi yoksa altında mı olduğunu (`income`: `<=50K` / `>50K`) tahmin eden bir **Random Forest** sınıflandırıcısı kuruyor. Veri seti yaş, eğitim, medeni hal, meslek, haftalık çalışma saati gibi hem sayısal hem kategorik özellikler içeriyor.

Tek bir Karar Ağacı yerine bir **topluluk (ensemble)** yöntemi kullanılmasının sebebi, çok sayıda ağacın oylarının ortalamasının tek bir ağaca göre daha kararlı ve genellenebilir sonuçlar vermesi. Notebook; veri temizleme, kategorik değişkenlerin `ColumnTransformer` ile kodlanması, sınıf dengesizliğine dikkat edilerek yapılan train/test ayrımı ve son olarak model değerlendirmesi + özellik önem dereceleri analizini kapsıyor.

**🇬🇧 English:**
This notebook builds a **Random Forest** classifier on the UCI Adult / Census Income dataset to predict whether a person's annual income is above or below $50K (`income`: `<=50K` / `>50K`). The dataset mixes numeric features (age, hours worked) with categorical ones (education, marital status, occupation).

An ensemble method is used instead of a single Decision Tree because averaging the votes of many trees tends to generalize better and reduce variance compared to any one tree. The notebook covers data cleaning, encoding categorical features with a `ColumnTransformer`, a stratified train/test split (given class imbalance), and finally model evaluation plus a feature-importance analysis.

## 📦 0. Kütüphaneler / Imports

**🇹🇷 Türkçe:**
Veri işleme, görselleştirme ve modelleme için gerekli temel kütüphaneler tek hücrede toplanıyor.

**🇬🇧 English:**
The core libraries needed for data handling, visualization, and modeling are collected in a single cell.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📥 1. Veri Setini Yükleme / Load the Dataset

**🇹🇷 Türkçe:**
Income Evaluation (Adult/Census Income) veri seti CSV dosyasından yükleniyor. Ham dosyada kolon adlarının başında boşluk karakteri var (`" workclass"` gibi) — bu, ileride kolon adlarıyla çalışırken dikkat gerektiren bir detay.

**🇬🇧 English:**
The Income Evaluation (Adult/Census Income) dataset is loaded from CSV. The raw column names carry a leading space (e.g. `" workclass"`) — a detail that needs handling before the column names can be used reliably.

In [2]:
df = pd.read_csv("../../Data/14-income_evaluation.csv")

## 🔍 2. Veriye İlk Bakış / First Look at the Data

**🇹🇷 Türkçe:**
`head()`, `isnull()`, `duplicated()`, `info()` ve `describe()` ile veri setinin genel yapısı inceleniyor. Bu aşamada 24 tam kopya (duplicate) satır tespit edilip siliniyor. `isnull()` henüz eksik değer göstermiyor çünkü eksik veriler bu veri setinde `NaN` değil, `"?"` string'i olarak kodlanmış — bunlar birazdan ayrıca ele alınacak.

**🇬🇧 English:**
`head()`, `isnull()`, `duplicated()`, `info()`, and `describe()` are used to inspect the overall shape of the dataset. 24 exact duplicate rows are found and dropped here. `isnull()` shows no missing values yet, because missing entries in this dataset aren't encoded as `NaN` but as the string `"?"` — those are handled separately below.

In [3]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
df.isnull().sum()

age                0
 workclass         0
 fnlwgt            0
 education         0
 education-num     0
 marital-status    0
 occupation        0
 relationship      0
 race              0
 sex               0
 capital-gain      0
 capital-loss      0
 hours-per-week    0
 native-country    0
 income            0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(24)

In [6]:
df = df.drop_duplicates()

In [7]:
df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.info()

<class 'pandas.DataFrame'>
Index: 32537 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              32537 non-null  int64
 1    workclass       32537 non-null  str  
 2    fnlwgt          32537 non-null  int64
 3    education       32537 non-null  str  
 4    education-num   32537 non-null  int64
 5    marital-status  32537 non-null  str  
 6    occupation      32537 non-null  str  
 7    relationship    32537 non-null  str  
 8    race            32537 non-null  str  
 9    sex             32537 non-null  str  
 10   capital-gain    32537 non-null  int64
 11   capital-loss    32537 non-null  int64
 12   hours-per-week  32537 non-null  int64
 13   native-country  32537 non-null  str  
 14   income          32537 non-null  str  
dtypes: int64(6), str(9)
memory usage: 4.0 MB


In [10]:
df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32537.000000,3.253700e+04,32537.000000,32537.000000,32537.000000,32537.000000
mean,38.585549,1.897808e+05,10.081815,1078.443741,87.368227,40.440329
std,13.637984,1.055565e+05,2.571633,7387.957424,403.101833,12.346889
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.369930e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [11]:
df.columns

Index(['age', ' workclass', ' fnlwgt', ' education', ' education-num',
       ' marital-status', ' occupation', ' relationship', ' race', ' sex',
       ' capital-gain', ' capital-loss', ' hours-per-week', ' native-country',
       ' income'],
      dtype='str')

## 🧹 3. Veri Temizleme / Data Cleaning

**🇹🇷 Türkçe:**
Bu bölümde birkaç ayrı temizlik adımı uygulanıyor:
- Kolon adlarındaki baştaki boşluklar `str.strip()` ile temizleniyor.
- `fnlwgt` (anket ağırlığı — kişiye özgü bir bilgi değil) ve `education` (sayısal karşılığı `education-num` zaten var, aynı bilginin tekrarı) kolonları modele katkısı olmadığı için çıkarılıyor.
- Kategorik kolonlardaki (`workclass` gibi) değerlerin başındaki boşluklar da `str.strip()` ile temizleniyor.
- Boşluklar temizlendikten sonra `"?"` değerleri gerçek eksik değer (`pd.NA`) olarak işaretleniyor ve `dropna()` ile bu satırlar veri setinden çıkarılıyor (`workclass`: 1836, `occupation`: 1843, `native-country`: 582 satır).

**🇬🇧 English:**
Several separate cleaning steps happen here:
- Leading whitespace in column names is stripped with `str.strip()`.
- `fnlwgt` (a survey weight, not a property of the person) and `education` (redundant with the already-numeric `education-num`) are dropped since they add no modeling value.
- Leading whitespace inside categorical values (e.g. `workclass`) is also stripped.
- Once whitespace is cleared, `"?"` values are marked as true missing values (`pd.NA`) and the affected rows are removed with `dropna()` (`workclass`: 1836, `occupation`: 1843, `native-country`: 582 rows).

In [12]:
df.columns = df.columns.str.strip()

In [13]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income'],
      dtype='str')

In [14]:
df = df.drop("fnlwgt", axis=1)

In [15]:
df = df.drop("education", axis=1)

In [16]:
df["workclass"].unique()

<StringArray>
[       ' State-gov', ' Self-emp-not-inc',          ' Private',
      ' Federal-gov',        ' Local-gov',                ' ?',
     ' Self-emp-inc',      ' Without-pay',     ' Never-worked']
Length: 9, dtype: str

In [17]:
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

C:\Users\efeka\AppData\Local\Temp\ipykernel_7916\1410376948.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns


In [18]:
df["workclass"].unique()

<StringArray>
[       'State-gov', 'Self-emp-not-inc',          'Private',
      'Federal-gov',        'Local-gov',                '?',
     'Self-emp-inc',      'Without-pay',     'Never-worked']
Length: 9, dtype: str

In [19]:
df = df.replace("?", pd.NA)

In [20]:
df.isnull().sum()

age                  0
workclass         1836
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     582
income               0
dtype: int64

In [21]:
df = df.dropna()

In [22]:
df.isnull().sum()

age               0
workclass         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

## 📊 4. Keşifsel Veri Analizi / Exploratory Data Analysis

**🇹🇷 Türkçe:**
Modele geçmeden önce hedef değişkenle ilişkili basit desenler aranıyor:
- `marital-status` bazında `>50K` oranı: `Married-civ-spouse` (%45.5) ve `Married-AF-spouse` (%47.6) gruplarında oran, `Never-married` (%4.8) grubuna göre **on kattan fazla** yüksek — medeni halin gelirle güçlü bir ilişkisi olduğunu gösteriyor.
- `income` gruplarına göre sayısal kolonların ortalaması: `>50K` grubunun ortalama yaşı (43.96), çalışma saati (45.71) ve eğitim yılı (11.61) hepsi `<=50K` grubundan belirgin şekilde yüksek.

**🇬🇧 English:**
Before modeling, simple patterns against the target are explored:
- `>50K` rate by `marital-status`: `Married-civ-spouse` (45.5%) and `Married-AF-spouse` (47.6%) show a rate **more than ten times** higher than `Never-married` (4.8%) — a strong signal that marital status relates to income.
- Mean of numeric columns by `income` group: the `>50K` group has a noticeably higher average age (43.96), hours worked (45.71), and years of education (11.61) than the `<=50K` group.

In [23]:
pd.crosstab(df['marital-status'], df['income'], normalize='index')

income,<=50K,>50K
marital-status,,
Divorced,0.892688,0.107312
Married-AF-spouse,0.523810,0.476190
Married-civ-spouse,0.544989,0.455011
Married-spouse-absent,0.916216,0.083784
Never-married,0.951601,0.048399
Separated,0.929712,0.070288
Widowed,0.903265,0.096735


In [24]:
df.groupby('income')[['age', 'hours-per-week', 'education-num']].mean()

,age,hours-per-week,education-num
income,,,
<=50K,36.611585,39.352008,9.630230
>50K,43.960165,45.707034,11.606981


In [25]:
df.head()

,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 🔤 5. Kategorik Değişkenleri İnceleme / Inspecting Categorical Columns

**🇹🇷 Türkçe:**
Encoding'e geçmeden önce her kategorik kolonun kaç farklı değer aldığı listeleniyor. `native-country` 41 farklı değerle en yüksek kardinaliteye sahip — bu, one-hot encoding sonrası kolon sayısını önemli ölçüde artıracak bir detay.

**🇬🇧 English:**
Before encoding, the distinct values of each categorical column are listed. `native-country` has the highest cardinality with 41 distinct values — a detail that will noticeably inflate the column count after one-hot encoding.

In [26]:
cat_cols = ["workclass","marital-status","occupation","relationship","race","sex","native-country"]

In [27]:
for i in cat_cols:
    print("\n")
    print(i, "\n")
    print(df[i].unique())



workclass 

<StringArray>
[       'State-gov', 'Self-emp-not-inc',          'Private',
      'Federal-gov',        'Local-gov',     'Self-emp-inc',
      'Without-pay']
Length: 7, dtype: str


marital-status 

<StringArray>
[        'Never-married',    'Married-civ-spouse',              'Divorced',
 'Married-spouse-absent',             'Separated',     'Married-AF-spouse',
               'Widowed']
Length: 7, dtype: str


occupation 

<StringArray>
[     'Adm-clerical',   'Exec-managerial', 'Handlers-cleaners',
    'Prof-specialty',     'Other-service',             'Sales',
  'Transport-moving',   'Farming-fishing', 'Machine-op-inspct',
      'Tech-support',      'Craft-repair',   'Protective-serv',
      'Armed-Forces',   'Priv-house-serv']
Length: 14, dtype: str


relationship 

<StringArray>
['Not-in-family', 'Husband', 'Wife', 'Own-child', 'Unmarried',
 'Other-relative']
Length: 6, dtype: str


race 

<StringArray>
['White', 'Black', 'Asian-Pac-Islander', 'Amer-Indian-Eskimo', 'O

## 🏗️ 6. Özellik Kodlama / Feature Encoding

**🇹🇷 Türkçe:**
Bu veri setindeki kategorik kolonların hepsi **nominal** (sıralaması olmayan) — yani `White`, `Black`, `Asian-Pac-Islander` gibi değerler arasında doğal bir "büyüklük" sırası yok. Bu yüzden çok sınıflı nominal kolonlar (`workclass`, `marital-status`, `occupation`, `relationship`, `race`, `native-country`) için `OneHotEncoder` kullanılıyor; her kategori kendi ikili (0/1) kolonuna dönüşüyor, böylece model kategoriler arasında var olmayan bir sıralama öğrenmiyor.

İkili (binary) `sex` kolonu için ise ayrı bir encoder'a gerek yok — `Male`/`Female` doğrudan `.map()` ile 1/0'a çevriliyor. Hedef değişken `income` de aynı şekilde `<=50K` → 0, `>50K` → 1 olarak sayısallaştırılıyor.

**⚠️ Önemli teknik not:** `ColumnTransformer`'ın varsayılan davranışı (`remainder="drop"`), listeye eklenmeyen tüm kolonları **sessizce** siler. Bu notebook'ta `one_hot_col` listesi sadece 6 kategorik kolonu içeriyor; `remainder="passthrough"` **açıkça** belirtilmezse `age`, `education-num`, `sex`, `capital-gain`, `capital-loss`, `hours-per-week` gibi sayısal kolonların hepsi modele hiç girmeden kaybolur — hiçbir hata mesajı vermeden. Aşağıdaki `ColumnTransformer` bu yüzden `remainder="passthrough"` ile tanımlanıyor.

**🇬🇧 English:**
Every categorical column in this dataset is **nominal** (unordered) — there's no natural ranking between values like `White`, `Black`, or `Asian-Pac-Islander`. That's why the multi-category nominal columns (`workclass`, `marital-status`, `occupation`, `relationship`, `race`, `native-country`) are encoded with `OneHotEncoder`: each category becomes its own binary (0/1) column, so the model never learns a false ordering between categories.

The binary `sex` column doesn't need a dedicated encoder — `Male`/`Female` is mapped directly to 1/0 with `.map()`. The target `income` is numerically encoded the same way: `<=50K` → 0, `>50K` → 1.

**⚠️ Important technical note:** `ColumnTransformer`'s default behavior (`remainder="drop"`) **silently drops** every column not listed in a transformer. Here, `one_hot_col` only lists 6 categorical columns; without explicitly setting `remainder="passthrough"`, numeric columns like `age`, `education-num`, `sex`, `capital-gain`, `capital-loss`, and `hours-per-week` would disappear from the model entirely — with no error raised. The `ColumnTransformer` below is therefore defined with `remainder="passthrough"`.

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder

In [29]:
one_hot_col = ["workclass", "marital-status", "occupation", "relationship", "race","native-country"]

In [30]:
one_hot_col

['workclass',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'native-country']

In [31]:
label_col = ["sex"]

In [32]:
one_hot_encoder_pipeline = Pipeline([("onehot", OneHotEncoder())])
one_hot_preprocessor = ColumnTransformer([("one_hot", one_hot_encoder_pipeline, one_hot_col)], remainder="passthrough")

In [33]:
df["income"].unique()

<StringArray>
['<=50K', '>50K']
Length: 2, dtype: str

In [34]:
df["income"] = df["income"].replace("<=50K", "0")
df["income"] = df["income"].replace(">50K","1")
df["income"] = df["income"].astype(int)

In [35]:
df["sex"] = df["sex"].map({"Male":1,"Female":0})

In [36]:
df

,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,1,2174,0,40,United-States,0
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,1,0,0,13,United-States,0
2,38,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,1,0,0,40,United-States,0
3,53,Private,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,1,0,0,40,United-States,0
4,28,Private,13,Married-civ-spouse,Prof-specialty,Wife,Black,0,0,0,40,Cuba,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,12,Married-civ-spouse,Tech-support,Wife,White,0,0,0,38,United-States,0
32557,40,Private,9,Married-civ-spouse,Machine-op-inspct,Husband,White,1,0,0,40,United-States,1
32558,58,Private,9,Widowed,Adm-clerical,Unmarried,White,0,0,0,40,United-States,0
32559,22,Private,9,Never-married,Adm-clerical,Own-child,White,1,0,0,20,United-States,0


## ✂️ 7. Train/Test Ayrımı / Train-Test Split

**🇹🇷 Türkçe:**
Hedef değişken (`income`) dengesiz bir dağılıma sahip (%76 `<=50K`, %24 `>50K`), bu yüzden `train_test_split` çağrısında `stratify=y` kullanılıyor — böylece eğitim ve test kümelerindeki sınıf oranları orijinal veri setiyle aynı kalıyor.

**🇬🇧 English:**
The target (`income`) is imbalanced (~76% `<=50K`, ~24% `>50K`), so `stratify=y` is used in `train_test_split` — this keeps the class ratio in both the train and test sets consistent with the original dataset.

In [37]:
from sklearn.model_selection import train_test_split

In [38]:
X = df.drop("income", axis=1)
y = df["income"]

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## 🌲 8. Ön İşleme ve Random Forest Modeli / Preprocessing & Random Forest Model

**🇹🇷 Türkçe:**
`ColumnTransformer`, **sadece eğitim kümesine** `fit_transform` ile uydurulup (fit) test kümesine yalnızca `transform` uygulanıyor — bu, test verisinin encoding istatistiklerine (örneğin hangi kategorilerin var olduğuna) sızmasını (data leakage) önlüyor. Kodlama sonrası özellik sayısı 12'den **86**'ya çıkıyor (6 nominal kolonun one-hot açılımı + 6 sayısal/ikili kolonun `passthrough` ile aynen aktarılması).

Ardından varsayılan hiperparametrelerle bir `RandomForestClassifier` eğitiliyor — bu notebook'ta hiperparametre araması (`GridSearchCV`) yapılmıyor, sonuçlar ağacın kendi varsayılan ayarlarıyla elde ediliyor.

**🇬🇧 English:**
The `ColumnTransformer` is fit **only on the training set** with `fit_transform`, while the test set only gets `transform` — this prevents test-set information (e.g. which categories exist) from leaking into the encoding. After encoding, the feature count grows from 12 to **86** (the one-hot expansion of 6 nominal columns plus the 6 numeric/binary columns carried through unchanged via `passthrough`).

A `RandomForestClassifier` is then trained with its default hyperparameters — no `GridSearchCV` tuning is performed in this notebook, so the results reflect the model's out-of-the-box settings.

In [40]:
X_train_enc = one_hot_preprocessor.fit_transform(X_train)
X_test_enc = one_hot_preprocessor.transform(X_test)

In [41]:
X_train_enc.shape

(22604, 86)

In [42]:
X_test_enc

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 73917 stored elements and shape (7535, 86)>

In [43]:
from sklearn.ensemble import RandomForestClassifier

In [44]:
rand = RandomForestClassifier()
rand.fit(X_train_enc, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [45]:
y_pred = rand.predict(X_test_enc)

## 📈 9. Model Değerlendirme / Model Evaluation

**🇹🇷 Türkçe:**
Sınıf dengesizliği yüzünden sadece accuracy'e bakmak yanıltıcı olabileceğinden, confusion matrix ve sınıf bazlı precision/recall/F1 birlikte inceleniyor:
- **Accuracy: %84.7**
- `<=50K` sınıfı (çoğunluk): precision 0.88, recall 0.92, F1 0.90
- `>50K` sınıfı (azınlık): precision 0.72, recall 0.63, F1 0.67

Azınlık sınıfındaki (`>50K`) daha düşük recall (0.63), modelin yüksek gelirli kişilerin bir kısmını (699 kişi) `<=50K` olarak yanlış sınıflandırdığını gösteriyor — bu, sınıf dengesizliğinin beklenen bir sonucu.

**🇬🇧 English:**
Because the classes are imbalanced, accuracy alone can be misleading, so the confusion matrix and per-class precision/recall/F1 are examined together:
- **Accuracy: 84.7%**
- `<=50K` class (majority): precision 0.88, recall 0.92, F1 0.90
- `>50K` class (minority): precision 0.72, recall 0.63, F1 0.67

The lower recall on the minority class (`>50K`, 0.63) shows the model misclassifies a meaningful share of high earners (699 people) as `<=50K` — an expected consequence of the class imbalance.

In [46]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [47]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))

[[5213  445]
 [ 696 1181]]
              precision    recall  f1-score   support

           0       0.88      0.92      0.90      5658
           1       0.73      0.63      0.67      1877

    accuracy                           0.85      7535
   macro avg       0.80      0.78      0.79      7535
weighted avg       0.84      0.85      0.84      7535

0.8485733244857332


## 🔎 10. Özellik Önem Dereceleri / Feature Importances

**🇹🇷 Türkçe:**
`rand.feature_importances_` dizisi 86 elemanlı (kodlanmış özellik sayısı kadar) olduğu için, orijinal `X_train.columns` (12 kolon) yerine `one_hot_preprocessor.get_feature_names_out()` ile isimlendiriliyor — aksi halde uzunluklar eşleşmediği için hata alınırdı.

Sıralanmış sonuçlara göre en etkili 5 özellik: **`age`** (%22.7), **`education-num`** (%12.9), **`hours-per-week`** (%11.4), **`capital-gain`** (%10.7) ve tek bir kategori olarak **`marital-status = Married-civ-spouse`** (%8.0). Dikkat çekici nokta: en etkili tüm özellikler `remainder="passthrough"` ile korunan **sayısal** kolonlar — bu da bir önceki bölümdeki `remainder` uyarısının neden kritik olduğunu somut biçimde gösteriyor: bu kolonlar yanlışlıkla silinseydi, model en güçlü sinyallerinden mahrum kalırdı.

**🇬🇧 English:**
`rand.feature_importances_` has 86 entries (matching the encoded feature count), so it's labeled with `one_hot_preprocessor.get_feature_names_out()` rather than the original `X_train.columns` (12 columns) — using the latter would raise a length-mismatch error.

The top 5 features by importance are: **`age`** (22.7%), **`education-num`** (12.9%), **`hours-per-week`** (11.4%), **`capital-gain`** (10.7%), and a single category, **`marital-status = Married-civ-spouse`** (8.0%). Notably, every one of the top features is a **numeric** column preserved via `remainder="passthrough"` — a concrete illustration of why the `remainder` warning in the encoding section mattered: had those columns been silently dropped, the model would have lost its strongest signals.

In [48]:
rand.feature_importances_

array([5.94934245e-03, 6.14488628e-03, 1.06474349e-02, 6.08741246e-03,
       8.48288227e-03, 5.09326112e-03, 6.45659968e-05, 9.79143615e-03,
       3.70276626e-04, 5.21537468e-02, 8.98415728e-04, 3.99802554e-02,
       2.60621998e-03, 2.11453244e-03, 5.90644949e-03, 7.39451856e-06,
       7.15603875e-03, 1.75564765e-02, 5.77556249e-03, 3.77891938e-03,
       4.11098430e-03, 8.19181445e-03, 1.57610790e-04, 1.82291426e-02,
       3.31281173e-03, 6.66513774e-03, 4.92353595e-03, 4.93992778e-03,
       4.51631286e-02, 1.27445197e-02, 1.80534237e-03, 7.52412073e-03,
       7.40136920e-03, 1.03882920e-02, 1.66788632e-03, 3.19667372e-03,
       4.79404397e-03, 1.04770822e-03, 6.58710936e-03, 3.45126712e-04,
       1.29096267e-03, 4.69801705e-04, 2.97087669e-04, 7.73847793e-04,
       2.33500261e-04, 1.08583902e-04, 2.40243555e-04, 1.07230169e-03,
       2.94556479e-04, 1.43495183e-03, 4.69751437e-04, 1.15968827e-04,
       2.09913723e-04, 1.76605323e-06, 8.06313051e-06, 1.80594078e-04,
      

In [49]:
pd.Series(rand.feature_importances_, index=one_hot_preprocessor.get_feature_names_out()).sort_values()

one_hot__native-country_Holand-Netherlands            0.000002
one_hot__occupation_Armed-Forces                      0.000007
one_hot__native-country_Honduras                      0.000008
one_hot__native-country_Outlying-US(Guam-USVI-etc)    0.000055
one_hot__workclass_Without-pay                        0.000065
                                                        ...   
one_hot__marital-status_Married-civ-spouse            0.052154
remainder__capital-gain                               0.110231
remainder__hours-per-week                             0.112184
remainder__education-num                              0.132839
remainder__age                                        0.226430
Length: 86, dtype: float64

## ✅ 11. Sonuç / Conclusion

**🇹🇷 Türkçe:**
Varsayılan hiperparametrelerle eğitilen Random Forest modeli, income tahmininde **%84.7 accuracy** ve azınlık sınıfında (`>50K`) 0.67 F1-skoru elde etti. `age`, `education-num`, `hours-per-week` ve `capital-gain` gibi sayısal özellikler modelin en güçlü sinyalleri oldu. Bu notebook'ta hiperparametre araması yapılmadı; `n_estimators`, `max_depth` ve `min_samples_leaf` gibi parametrelerin `GridSearchCV` ile aranması, özellikle azınlık sınıfındaki recall'u artırmak için doğal bir sonraki adım.

**🇬🇧 English:**
The Random Forest model, trained with default hyperparameters, reached **84.7% accuracy** on income prediction and an F1-score of 0.67 on the minority class (`>50K`). Numeric features — `age`, `education-num`, `hours-per-week`, and `capital-gain` — turned out to be the strongest signals. No hyperparameter search was performed in this notebook; tuning `n_estimators`, `max_depth`, and `min_samples_leaf` via `GridSearchCV` is a natural next step, particularly to improve recall on the minority class.